In [7]:
#H2 results

from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver

h2_driver = PySCFDriver(
    atom=[
        "H  0.000  0.000  0.000",
        "H  0.000  0.000  0.7414"
    ],
    unit=DistanceUnit.ANGSTROM,
    basis="sto3g",
    charge=0,
    spin=0
)

h2_problem = h2_driver.run()

print("Electrons:", h2_problem.num_particles)
print("Spatial orbitals:", h2_problem.num_spatial_orbitals)
print("Nuclear repulsion energy:", h2_problem.nuclear_repulsion_energy)

Electrons: (1, 1)
Spatial orbitals: 2
Nuclear repulsion energy: 0.7137539936876182


In [8]:
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
from qiskit_nature.second_q.mappers import JordanWignerMapper

# Create the active-space model
h2_transformer = ActiveSpaceTransformer(
    num_electrons=2,
    num_spatial_orbitals=2
)

h2_active_problem = h2_transformer.transform(h2_problem)

# Convert the chemistry Hamiltonian into qubits
h2_mapper = JordanWignerMapper()

h2_hamiltonian = h2_mapper.map(
    h2_active_problem.hamiltonian.second_q_op()
)

print("Electrons:", h2_active_problem.num_particles)
print("Spatial orbitals:", h2_active_problem.num_spatial_orbitals)
print("Qubits:", h2_hamiltonian.num_qubits)

Electrons: (1, 1)
Spatial orbitals: 2
Qubits: 4


In [9]:
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD

# Create the Hartree-Fock starting state
h2_hf = HartreeFock(
    num_spatial_orbitals=h2_active_problem.num_spatial_orbitals,
    num_particles=h2_active_problem.num_particles,
    qubit_mapper=h2_mapper
)

# Create the UCCSD circuit
h2_ansatz = UCCSD(
    num_spatial_orbitals=h2_active_problem.num_spatial_orbitals,
    num_particles=h2_active_problem.num_particles,
    qubit_mapper=h2_mapper,
    initial_state=h2_hf
)

print("Number of qubits:", h2_ansatz.num_qubits)
print("Number of parameters:", h2_ansatz.num_parameters)
print("Circuit depth:", h2_ansatz.depth())

Number of qubits: 4
Number of parameters: 3
Circuit depth: 1


In [10]:
from qiskit_algorithms import NumPyMinimumEigensolver

h2_exact_solver = NumPyMinimumEigensolver()

h2_exact_result = h2_exact_solver.compute_minimum_eigenvalue(
    h2_hamiltonian
)

h2_classical_energy = float(
    h2_exact_result.eigenvalue.real
)

print(
    "H2 classical exact electronic energy:",
    h2_classical_energy,
    "Hartree"
)

print("Hamiltonian constants:")
print(h2_active_problem.hamiltonian.constants)

H2 classical exact electronic energy: -1.8510241683485193 Hartree
Hamiltonian constants:
{'nuclear_repulsion_energy': np.float64(0.7137539936876182), 'ActiveSpaceTransformer': np.float64(0.0)}


In [11]:
from getpass import getpass
from qiskit_ibm_runtime import QiskitRuntimeService

ibm_token = getpass("Enter your IBM Quantum API key: ")

service = QiskitRuntimeService(
    channel="ibm_quantum_platform",
    token=ibm_token
)

backend = service.backend("ibm_fez")

print("Connected to:", backend.name)

qiskit_runtime_service._resolve_cloud_instances:WARNING:2026-11-07 14:12:07,076: Default instance not set. Searching all available instances.


Connected to: ibm_fez


In [12]:
from qiskit.transpiler.preset_passmanagers import (
    generate_preset_pass_manager
)

pm_optimized = generate_preset_pass_manager(
    backend=backend,
    optimization_level=3
)

h2_isa_ansatz = pm_optimized.run(h2_ansatz)

# Map the Hamiltonian to the physical qubits
h2_isa_hamiltonian = h2_hamiltonian.apply_layout(
    h2_isa_ansatz.layout
)

print("Physical circuit width:", h2_isa_ansatz.num_qubits)
print("Circuit depth:", h2_isa_ansatz.depth())
print("Gate counts:", h2_isa_ansatz.count_ops())
print("Number of parameters:", h2_isa_ansatz.num_parameters)

Physical circuit width: 156
Circuit depth: 160
Gate counts: OrderedDict({'rz': 90, 'sx': 82, 'cz': 49, 'x': 6})
Number of parameters: 3


In [13]:
import numpy as np
from qiskit_ibm_runtime import EstimatorV2 as Estimator

# Set up the IBM estimator
estimator = Estimator(mode=backend)
estimator.options.default_shots = 4096
estimator.options.resilience_level = 1

# Start with all UCCSD parameters at zero
h2_initial_parameters = np.zeros(
    h2_isa_ansatz.num_parameters
)

print("Submitting H2 calculation...")

h2_job = estimator.run([
    (
        h2_isa_ansatz,
        h2_isa_hamiltonian,
        h2_initial_parameters
    )
])

print("Job submitted!")
print("Job ID:", h2_job.job_id())

Submitting H2 calculation...
Job submitted!
Job ID: dbnj3g8ou4as73b4lvv0


In [16]:
print(h2_job.status())

QUEUED


In [3]:
h2_result = h2_job.result()

h2_hardware_energy = float(
    h2_result[0].data.evs
)

h2_energy_offset = sum(
    float(value)
    for value in h2_active_problem.hamiltonian.constants.values()
)

h2_hardware_total = (
    h2_hardware_energy + h2_energy_offset
)

print("IBM electronic energy:", h2_hardware_energy, "Hartree")
print("IBM total energy:", h2_hardware_total, "Hartree")

print(
    "Classical total energy:",
    h2_classical_energy + h2_energy_offset,
    "Hartree"
)

print(
    "Absolute difference:",
    abs(h2_hardware_energy - h2_classical_energy),
    "Hartree"
)

NameError: name 'h2_job' is not defined